# Hierarchical Mixed Effects Modeling of ROI and Rater Covariances for the Dorsal Visual Cortex

## Header

This section configures the notebook, imports dependencies, and declares utility functions for use in the rest of the notebook.

The first time you use this notebook, make sure the `Configuration` section is set up correctly for your system.

### Configuration

#### Paths

In [ ]:
from pathlib import Path

# The cache directories:
cache_path = Path('/data')

# The processing path and subpaths:
data_path = Path('/data/crcns2021/results/proc')

# The file of visual surface areas for the data.
ventral_sarea_path = data_path / 'ventral_sareas.tsv'

### Dependencies

In [ ]:
import os, sys, json

import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipyvolume as ipv
import neuropythy as ny
import pandas as pd
import torch, pimms
import torch.distributions as dist

import hcpannot

In [ ]:
%matplotlib inline

In [ ]:
# Additional matplotlib preferences:
font_data = {'family':'sans-serif',
             'sans-serif':['HelveticaNeue', 'Helvetica', 'Arial'],
             'size': 10,
             'weight': 'light'}
mpl.rc('font',**font_data)
# we want relatively high-res images, especially when saving to disk.
mpl.rcParams['figure.dpi'] = 72*2
mpl.rcParams['savefig.dpi'] = 72*4
# Fix the sometimes issue of matplotlib not drawing minus signs correctly.
mpl.rcParams['axes.unicode_minus'] = False

#### Configure `hcpannot`

In [ ]:
# Note the cache path we want to use outside the docker container;
# normally this gets set by the Docker startup, so here outside
# the docker container we set it manually.
hcpannot.interface.default_load_path = str(cache_path)
# We have to update some data after setting this.

# The Subject List:
from hcpannot.config import subject_list
# The subject IDs we are processing over, as a numpy array.
sids = np.array(subject_list)

# We redefine proc to give it useful default parameters.
def proc(*, data_path=data_path, **kw):
    from hcpannot.proc import proc as _proc
    return _proc('dorsal', **kw, data_path=data_path)

## Model of the Covariance of ROI Sizes

### Data

Here we load the data from the processed data sources into a 
dataframe, `ventdf_meas` (ventral dataframe: measurements).

In [ ]:
dorsal_raters = hcpannot.config.raters_by_region['dorsal']

data = {
    (rater, sid, h): proc(rater=rater, sid=sid, hemisphere=h, overwrite=False)
    for rater in dorsal_raters
    for sid in sids
    for h in ('lh','rh')}

rows = []
for ((rater, sid, h), ent) in data.items():
    row = dict(rater=rater, sid=sid, hemisphere=h)
    try:
        ent = ent['reports']['surface_area']
    except Exception as e:
        ent = dict(cortex=np.nan, V3a=np.nan, V3b=np.nan, LO1=np.nan, IPS0=np.nan)
    for (k,v) in ent.items():
        row[k + '_mm2'] = v
        row[k + '_percent'] = v * 100 / ent['cortex']
    rows.append(row)

dorsdf_meas = pd.DataFrame(rows)
dorsdf_meas

### Model

Here we declare our model, `ROISizesModel` which attempts to understand the true sizes of the areas and the correlations between them based on the many ratings made of each individual subject's left and right hemisphere ventral ROIs.

There are:
 - 6 raters
 - 181 subjects
 - 8 ROIs (2 hemispheres with 4 ROIs each)

To explain the model, we'll use the following notation:
 - $z_{r,s,k}$ is the *measured* size of the **$k$th ROI** (1: LHV4; 2: LVO1; 3: LVO2; 4: RHV4; 5: RVO1; 6: RVO2),
   the **$s$th subject** (1&ndash;181),
   as drawn by **rater $r$** (1&ndash;5).
 - $\hat{z}_{r,s,k}$ is the *predicted* size of the annotation for ROI $k$ made by rater $r$ in subject $s$.
 - $\zeta_{s,k}$ is the (unknown) true size of every ROI in every subject.
 - $\hat{\zeta}_{s,k}$ is the model's prediction of the true ROI size ($\hat{\zeta}_{s,k}$ is a prediction of $\zeta_{s,k}$).
 - $\mathcal{Z}_k$ is the random variable representing the distribution of the size of ROI $k$ in the world; i.e., the values of $\zeta_{:,k}$ for each subject are distributed according to the $\mathcal{Z}_k$ random variable.
 - ${Z}_{r,k}$ is the random variable representing the distribution of tsize of ROI $k$ as drawn by rater $r$; i.e., the values of $z_{r,:.k}$ for each subject are distributed according to the $Z_{r,k}$ random variable.

Note that $\mathcal{Z}_k$ and ${Z}_{r,k}$ are different&mdash;the former represents the distribution of the *true* sizes of the ROIs in the world, while the latter represents the distribution of sizes that would be drawn by rater $r$.

We propose the following model:
 - **Goal**. 
   Our fundamental goal is to estimate the true variance and covariance of each and each pair
   in order to estimate correlations that have been adjusted for measurement noise.
   Note that each subject is a single observation of all 6 values.
 - For each subject in our dataset there are 6 *true* ROI sizes, and we assume that these sizes can be modeled according to a multivariate normal distribution over the vector of ROI sizes:
   
   $$
   \begin{pmatrix}
     \mathcal{Z}_{\text{LHV4}} \\
     \mathcal{Z}_{\text{LVO1}} \\
     \mathcal{Z}_{\text{LVO2}} \\
     \mathcal{Z}_{\text{RHV4}} \\
     \mathcal{Z}_{\text{RVO1}} \\
     \mathcal{Z}_{\text{RVO2}} \\
   \end{pmatrix} \sim \mathcal{N}\left(\boldsymbol{\mu_\mathcal{Z}}, \mathbf{\Sigma_\mathcal{Z}}\right)
   $$
   
   Where $\boldsymbol{\mu_\mathcal{Z}}$ is 6D vector of the mean sizes and $\mathbf{\Sigma_\mathcal{Z}}$ is the covariance matrix:

   $$
   \mathcal{N}\left(\boldsymbol{\mu_\mathcal{Z}}, \mathbf{\Sigma_\mathcal{Z}}\right) = \mathcal{N}\left(
      \begin{pmatrix}
         \mu_\text{LHV4} \\
         \mu_\text{LVO1} \\
         \mu_\text{LVO2} \\
         \mu_\text{RHV4} \\
         \mu_\text{RVO1} \\
         \mu_\text{RVO2} \\
      \end{pmatrix},
      \begin{pmatrix}
         \sigma_\text{LHV4}^2 + \delta
             & \sigma_\text{LHV4,LVO1}
             & \sigma_\text{LHV4,LVO2}
             & \sigma_\text{LHV4,RHV4}
             & \sigma_\text{LHV4,RVO1}
             & \sigma_\text{LHV4,RVO2} \\
         \sigma_\text{LHV4,LVO1}
             & \sigma_\text{LVO1}^2 + \delta
             & \sigma_\text{LVO1,LVO2}
             & \sigma_\text{LVO1,RHV4}
             & \sigma_\text{LVO1,RVO1}
             & \sigma_\text{LVO1,RVO2} \\
         \sigma_\text{LHV4,LVO2}
             & \sigma_\text{LVO1,LVO2}
             & \sigma_\text{LVO2}^2 + \delta
             & \sigma_\text{LVO2,RHV4}
             & \sigma_\text{LVO2,RVO1}
             & \sigma_\text{LVO2,RVO2} \\
         \sigma_\text{LHV4,RHV4}
             & \sigma_\text{LVO1,RHV4}
             & \sigma_\text{LVO2,RHV4}
             & \sigma_\text{RHV4}^2 + \delta
             & \sigma_\text{RHV4,RVO1}
             & \sigma_\text{RHV4,RVO2} \\
         \sigma_\text{LHV4,RVO1}
             & \sigma_\text{LVO1,RVO1}
             & \sigma_\text{LVO2,RVO1}
             & \sigma_\text{RHV4,RVO1}
             & \sigma_\text{RVO1}^2 + \delta
             & \sigma_\text{RVO1,RVO2} \\
         \sigma_\text{LHV4,RVO2}
             & \sigma_\text{LVO1,RVO2}
             & \sigma_\text{LVO2,RVO2}
             & \sigma_\text{RHV4,RVO2}
             & \sigma_\text{RVO1,RVO2}
             & \sigma_\text{RVO2}^2 + \delta
   \end{pmatrix}
   \right)
   $$

   In the above vector and matrix, the $\mu_k$ and $\sigma_k^2$ represent the mean and variance of the $k$th ROI,
   and the $\sigma_{j,k}$ values represent covariances between areas $j$ and $k$. The $\delta$ is a small value
   that prevents the variance from going to zero (which would be invalid). This should typically be many orders of
   magnitude smaller than the minimum variance (for square mm, we use 2.5e-2 and for percent, we use 2.5e-6).
 - (If we can estimate the values of $\mu$ and $\sigma$ in the matrix above, then the correlation
   $\rho_{j,k} = \sigma_{j,k} / \sqrt{\sigma_j^2 \sigma_k^2}$.)
 - The above distribution deals with the *true* sizes of the ROIs, which we do not have direct access to. It says, essentially, that across
   the whole population, the distribution is as described. In our case, we don't have simple 6D measurements of each subject;
   rather, we have 6D measurements of each subject *by each rater*. Since we have 5 raters, we really have 30 measurements per subject:
   6 ROIs for rater 1, 6 ROIs for rater 2 ... 6 ROIs for rater 5.
 - We assume that each rater $r$ had an additive bias for each ROI, $b_{r,k}$. We assume that these
   values are fixed effects that sum to 0 (this is a "fixed effect" of the model because we treat the biases as free parameters with no prior distribution).
 - The model assumes the annotated sizes made by each rater and ROI ($\hat{z}_{r,:,k}$) will be normally distributed also.
   Since each rater has a known bias for each ROI, we can assume that their mean for an ROI is the true mean for the ROI plus that bias ($\hat{\zeta}_{s,k} + b_{r,k}$);
   additionally, we expect each rater's annotated sizes to have their own normally distributed noise distribution around this mean with a variance of $\tau_{r,k}^2$.
   
   $$ \hat{z}_{r,:,k} \sim \mathcal{N}(\hat{\zeta}_{s,k} + b_{r,k}, \tau_{r,k}^2) $$
 - To represent the 30-dimensional observation vector of all 6 ROIs drawn by all 5 raters, we get

   $$
   \begin{pmatrix}
     Z_{\text{rater1},\text{LHV4}} \\
     Z_{\text{rater1},\text{LVO1}} \\
     Z_{\text{rater1},\text{LVO2}} \\
     Z_{\text{rater1},\text{RHV4}} \\
     Z_{\text{rater1},\text{RVO1}} \\
     Z_{\text{rater1},\text{RVO2}} \\
     Z_{\text{rater2},\text{LHV4}} \\
     \vdots \\
     Z_{\text{rater6},\text{RVO2}}    
   \end{pmatrix} \sim \mathcal{N}\left(\boldsymbol{\mu_Z}, \mathbf{\Sigma_Z}\right)
   $$

   (Note that we distinguish between $(\boldsymbol{\mu_\mathcal{Z}}, \mathbf{\Sigma_\mathcal{Z}})$ and $(\boldsymbol{\mu_{Z}}, \mathbf{\Sigma_{Z}})$;
   the former are the 6D parameters for the distribution of an individual subject and the latter are the 30D parameters for the distribution of our
   group of raters.)
 - Given this distribution we can perform a maximum likelihood estimation of the distribution parameters from our measurements. Our likelihood function is:

   $$
   \begin{eqnarray}
   \mathcal{P}\left(\boldsymbol{z} | \boldsymbol{\mu_Z},\mathbf{\Sigma_Z},\boldsymbol{b},\boldsymbol{\tau}\right)
       &=& \prod_s \mathcal{P}_{\mathcal{N}(\boldsymbol{\mu_Z}, \mathbf{\Sigma_Z})}(\boldsymbol{\hat{z}_{\cdot,s,\cdot}})
   \end{eqnarray}
   $$

   Where $\boldsymbol{\hat{z}_{\cdot,s,\cdot}}$ is taken to be the 30D vector of raters and ROIs for subject $s$.
   We minimize the negative log likelihood to get:

   $$
   \begin{eqnarray}
   \min_{\boldsymbol{\mu_Z},\mathbf{\Sigma_Z},\mathbf{b},\mathbf{\tau}} 
       \log\left( \prod_s \mathcal{P}_{\mathcal{N}(\boldsymbol{\mu_Z}, \mathbf{\Sigma_Z})}(\boldsymbol{\hat{z}_{\cdot,s,\cdot}})\right) \\
   \min_{\boldsymbol{\mu_Z},\mathbf{\Sigma_Z},\mathbf{b},\mathbf{\tau}}
       \sum_s \log\left( \mathcal{P}_{\mathcal{N}(\boldsymbol{\mu_Z}, \mathbf{\Sigma_Z})}(\boldsymbol{\hat{z}_{\cdot,s,\cdot}})\right)
   \end{eqnarray}
   $$

   Note that $\log\left(\mathcal{P}_{\mathcal{N}(\boldsymbol{\mu_Z}, \mathbf{\Sigma_Z})}(\boldsymbol{x})\right)$ can be calculated in PyTorch
   using the `torch.distributions.MultivariateNormal` class's `log_prob` method.
 - Critically, the 30x30 matrix $\mathbf{\Sigma_Z}$ has a clear relationship to the much smaller 6x6 matrix $\mathbf{\Sigma_\mathcal{Z}}$.
   For any two raters $r$ and $a$ ($a \ne r$), we assume their ratings are independent; thus the (6x6) covariance matrix of the
   6 ROIs as drawn by $r$ compared to the 6 ROIs as drawn by $a$ should simply be $\mathbf{\Sigma_\mathcal{Z}}$. For a single rater $r$,
   their covariance matrix of their 6 ROIs versus the same ROIs should be the same matrix, just with additional terms to represent the
   variance of their errors on each ROI. These values are variances of the ROIs, so they are added only to the diagonal (since the
   mean of their noise distribution is 0 for each ROI, there should be no effect on the off-diagonals). These values are the
    $\tau_{r,k}^2$ parameters.

   So in effect, the covariance of the entire 30x30 per-subject all-measurements matrix is as follows.

   $$
   \mathbf{\Sigma_Z} = \begin{pmatrix}
      \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{1,:}^2) &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{2,:}^2) & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{3,:}^2) & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{4,:}^2) & \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{5,:}^2) \\ 
   \end{pmatrix}
   $$
 - Similarly, we know the mean vector $\boldsymbol{\mu_Z}$ to be made up of a stack of copies of the
   true mean vector across subjects ($\boldsymbol{\mu_\mathcal{Z}}$), one per rater, plus the rater's
   bias $\boldsymbol{b_{r,:}}$:

   $$
   \boldsymbol{\mu_Z} = \begin{pmatrix}
      \boldsymbol{\mu_\mathcal{Z}} + \boldsymbol{b_{1,:}} \\
      \boldsymbol{\mu_\mathcal{Z}} + \boldsymbol{b_{2,:}} \\
      \boldsymbol{\mu_\mathcal{Z}} + \boldsymbol{b_{3,:}} \\
      \boldsymbol{\mu_\mathcal{Z}} + \boldsymbol{b_{4,:}} \\
      \boldsymbol{\mu_\mathcal{Z}} + \boldsymbol{b_{5,:}} \\
   \end{pmatrix}
   $$
- Overall, this gives us just 81 parameters for our model:
  - 21 come from the covariance matrix $\mathbf{\Sigma_\mathcal{Z}}$; there are 36 entries, but the matrix must be
    positive definite to be a valid covariance matrix, which includes requiring its symmetry.
    A 6x6 symmetric matrix has only 21 unique entries. (In the python code below we construct
    the covariance matrix from its Cholesky decomposition, in which our parameters are embedded.)
  - 6 come from the mean vector $\boldsymbol{\mu_\mathcal{Z}}$.
  - 30 come from $\mathbf{\tau_{r,k}}$ matrix of the variance of the noise that each rater has on each
    individual ROI (6 ROIs x 5 raters).
  - 24 come from the rater biases; the biases are constrained to sum to zero across raters, so there are only
    4 degrees of freedom from the raters instead of five. (In the python code below, we only store the biases
    for the first 4 raters, and we construct the last rater's bias as `-sum(other_rater_biases, axis=0)`.

In [ ]:
class ROISizesModel:
    """A model of the distribution of the true sizes of a group of ROIs drawn
    in a group of subjects by a group of raters.

    This class implements a hierarchical linear mixed effects model and
    uses maximum likelihood optimization to obtain an estimate of the
    covariance matrix and mean size of the population of all subjects based
    on a the drawings of a set of raters.

    Attributes
    ----------
    input_dataframe : pandas.DataFrame
        The dataframe that was passed to the model's constructor unaltered.
    suffix : str
        The suffix of columns from the ``input_dataframe`` that should be used
        as size measurements. The model assumes that columns will be named
        either ``'ROI_percent'`` or ``'ROI_mm2'`` (or similar) where the
        ``ROI`` is replaced with the ROI's name.
    dtype : torch.dtype
        The PyTorch ``dtype`` being used by the model.
    lr : float
        The learning rate of the model.
    max_iter : int
        The maximum number of LBFGS iterations.
    history_size : int
        The size of the LBFGS history.
    meas_dataframe : pandas.DataFrame
        The dataframe of measurements that has been reorganized by the model
        from the ``input_dataframe``.
    meas_size : torch.Tensor
        The ``S x RHK`` matrix of measured sizes extracted from the
        ``input_dataframe``. Note that the ``S`` dimension of this matrix may
        be less than the true number of subjects because subjects with any
        NaN values in any rater's ROI sizes are excluded.
    param_bias : torch.Tensor
        The bias parameter of the model. The model's bias parameter is
        represented as an ``R x K`` matrix where ``R`` is the number of raters
        and ``K`` is the number of ROIs in the dataset. The sum of the rows of
        the bias matrix is always a length ``K`` zero vector.
    param_mean : torch.Tensor
        A ``HK`` length vector of mean ROI sizes for the distribution of true 
        ROI sizes across subjects.
    param_fullmean : torch.Tensor
        A ``RHK`` length vector of mean ROI sizes for the distribution of true
        measured sizes from the raters.
    param_cov : torch.Tensor
        The covariance matrix parameter of the model. The model's true-size 
        covariance matrix is a ``HK x HK`` (where ``K`` is the number of ROIs
        in the dataset and ``H`` is the number of hemispheres) representation
        of the true covariance of the sizes of the ROIs with respect to each
        other. Note that this matrix is for the distribution of true ROIs in
        the world, not the representation of the dataset or a particular
        rater. The ``param_cov`` matrix is always positive definite.
    param_fullcov : torch.Tensor
        The full ``RHK x RHK`` covariance matrix parameter of the model. Where
        the ``param_cov`` is the ``HK x HK`` covariance matrix of the true ROI
        sizes of a subject, the ``param_fullcov`` is the covariance matrix of
        the sizes of the ROIs as annotated by our raters.
    param_rater_roi_logvar : torch.Tensor
        An ``R x HK`` matrix of the log of the variance of each rater's error
        distribution on each of the ``HK`` ROIs.
    pred_size : torch.tensor
        The true size of each subject's ROIs, as predicted by the model.
        The ``pred_size`` matrix has a shape of ``(S, HK)`` where ``S`` is the
        number of rows in the ``meas_size`` matrix, ``H`` is the number of
        hemispheres, and ``K`` is the number of ROIs.
    pred_dataframe : pandas.DataFrame
        A dataframe of the model's predictions.
    full_dataframe : pandas.DataFrame
        A copy of the ``meas_dataframe`` field with additional columns
        detailing the model's predictions.
    """
    _idcols = ('rater','sid','hemisphere')
    def __init__(self, df,
                 suffix='_percent',
                 dtype=torch.float32,
                 lr=0.01,
                 max_iter=2000,
                 history_size=100,
                 minvar=0.025,
                 fit=True):
        # Record the basic class parameters:
        self.input_dataframe = df
        self.suffix = suffix
        self.dtype = dtype
        self.lr = lr
        self.max_iter = max_iter
        self.minvar = torch.tensor(minvar)
        self.history_size = history_size
        # Organize the dataframe, params, and indices:
        idcols = list(type(self)._idcols)
        datacols = [
            k for k in df.columns
            if k.endswith(suffix) and not k.startswith('cortex')]
        df = df[idcols + datacols]
        suflen = len(suffix)
        df = df.rename(columns={c: c[:-suflen] for c in datacols})
        datacols = [c[:-suflen] for c in datacols]
        ii = np.all(np.isfinite(df[datacols].values), axis=1)
        df = df.iloc[ii, :]
        df = df.melt(
            id_vars=idcols,
            var_name='roi',
            value_name='size')
        self.meas_dataframe = df
        # Make indices of the dataframe's elements:
        idcols.append('roi')
        idcs = df[idcols].values
        sizes = torch.tensor(df['size'].values, dtype=dtype)
        nrows = len(sizes)
        (rtrs, sids, hems, rois) = idcs.T
        self.measvec_size = sizes
        uniqrtrs = np.unique(rtrs)
        uniqsids = np.unique(sids)
        uniqhems = np.unique(hems)
        uniqrois = np.unique(rois)
        nrtrs = len(uniqrtrs)
        nsids = len(uniqsids)
        nhems = len(uniqhems)
        nrois = len(uniqrois)
        self.counts = {'raters': nrtrs, 'sids': nsids, 'hems': nhems, 'rois': nrois}
        self.uniques = {
            'raters': uniqrtrs, 'sids': uniqsids,
            'hems': uniqhems, 'rois': uniqrois}
        self.rtr_index = np.zeros(nrows, dtype=int)
        self.sid_index = np.zeros(nrows, dtype=int)
        self.hem_index = np.zeros(nrows, dtype=int)
        self.roi_index = np.zeros(nrows, dtype=int)
        for (ii, rtr) in enumerate(uniqrtrs):
            self.rtr_index[rtrs == rtr] = ii
        for (ii, sid) in enumerate(uniqsids):
            self.sid_index[sids == sid] = ii
        for (ii, hem) in enumerate(uniqhems):
            self.hem_index[hems == hem] = ii
        for (ii, roi) in enumerate(uniqrois):
            self.roi_index[rois == roi] = ii
        # Process the minvar arg into a useful matrix:
        if minvar is None:
            self.minvar_diag = 0
        else:
            self.minvar_diag = torch.diag(
                torch.tensor([minvar]*nhems*nrois, dtype=dtype))
        # We need to organize our measured size data; for this, we'll need an
        # roi x hem x rater index:
        self._hemroi_index = self.roi_index + nrois*self.hem_index
        self._rtrhemroi_index = (
            + nrois*nhems * self.rtr_index
            + nrois * self.hem_index
            + self.roi_index)
        self.meas_size = torch.full(
            (nsids, nrtrs*nhems*nrois),
            torch.nan,
            dtype=dtype)
        self.meas_size[self.sid_index, self._rtrhemroi_index] = sizes
        # In the fitting, we'll exclude rows with NaNs:
        self._meas_okrows = ~torch.any(torch.isnan(self.meas_size), dim=1)
        # Set up the parameters for the model; first, the mean size and
        # covariance matrix for each ROI (\Sigma_\mathcal{Z} in the text
        # above).
        self.param_mean = torch.zeros(
            nhems * nrois,
            dtype=dtype)
        self._param_precov_logdiag = torch.ones(
            nhems*nrois,
            dtype=dtype,
            requires_grad=True)
        nels = nhems*nrois
        self._param_precov_offdiag = torch.zeros(
            nels*(nels - 1) // 2,
            dtype=dtype,
            requires_grad=True)
        self._cov_triu = tuple(torch.triu_indices(nels, nels, offset=1))
        self._cov_tril = tuple(torch.tril_indices(nels, nels, offset=-1))
        # The bias, which each rater has for each rater and roi:
        # nraters x nrois
        self._param_bias = torch.zeros(
            (nrtrs - 1, nrois),
            dtype=dtype,
            requires_grad=True)
        # The log of the variance (tau above), which we also have for each rater and roi:
        self.param_rater_roi_logvar = torch.zeros(
            (nrtrs, nrois),
            dtype=dtype)
        for (ii,roi) in enumerate(uniqrois):
            szs = sizes[rois == roi]
            mu = torch.mean(szs)
            self.param_mean[ii::nrois] = mu
            self.param_rater_roi_logvar[:,ii::nrois] = torch.log(torch.var(szs))
        self.param_mean.requires_grad_(True)
        self.param_rater_roi_logvar.requires_grad_(True)
        # Our final set of parameters (the real ones, not the derived but more
        # informative ones):
        self.params = {
            'bias': self._param_bias,
            'logvar': self.param_rater_roi_logvar,
            'mean': self.param_mean,
            'logcovdiag': self._param_precov_logdiag,
            'covoffdiag': self._param_precov_offdiag}
        # If requested, run the fit; otherwise, just return!
        self.hasfit = False
        if fit:
            self.fit()
    def fit(self, lr=None, max_iter=None, history_size=None):
        """Fit the model to the data that was initially given.

        This function is typically run automatically by the constructor, but
        if the ``fit=False`` parameter is given to the constructor, then
        this function must be run prior to using and of the model's
        predictions.
        """
        # Parameters (default to the ones given in the constructor):
        lr = self.lr if lr is None else lr
        max_iter = self.max_iter if max_iter is None else max_iter
        history_size = self.history_size if history_size is None else history_size
        self.lr = lr
        self.max_iter = max_iter
        self.history_size = history_size
        # Set up the optimizer:
        optim = torch.optim.LBFGS(
            self.params.values(),
            lr=lr,
            max_iter=max_iter,
            line_search_fn='strong_wolfe',
            history_size=history_size)
        # Since the max iterations is given in the LBFGS optimizer, we don't have
        # a loop here, just a closure and a step:
        meas_mtx = self.meas_size[self._meas_okrows, :]
        def closure():
            optim.zero_grad()
            d = dist.MultivariateNormal(
                loc=self.param_fullmean,
                covariance_matrix=self.param_fullcov)
            loss = -torch.sum(d.log_prob(meas_mtx))
            loss.backward()
            return loss
        optim.step(closure)
        self.hasfit = True
        return optim
    @property
    def param_bias(self):
        last = -self._param_bias.sum(dim=0, keepdim=True)
        return torch.cat((self._param_bias, last), dim=0)
    @property
    def param_fullmean(self):
        bias = self.param_bias
        nrtrs = bias.shape[0]
        bias = torch.flatten(torch.hstack([bias, bias]))
        return torch.cat([self.param_mean]*nrtrs) + bias
    @property
    def param_fullcov(self):
        rtr_roi_var = torch.exp(self.param_rater_roi_logvar)
        nrtrs = rtr_roi_var.shape[0]
        rtr_roi_var = torch.flatten(torch.hstack([rtr_roi_var, rtr_roi_var]))
        cov = torch.tile(self.param_cov, (nrtrs, nrtrs))
        return cov + torch.diag(rtr_roi_var)
    @property
    def param_cov(self):
        M = torch.diag(torch.exp(self._param_precov_logdiag))
        M[self._cov_tril] = self._param_precov_offdiag
        M = M @ M.T + self.minvar_diag
        return M
    @property
    def _paramvec_logvar(self):
        return self.param_rater_roi_logvar[self.rtr_index, self.roi_index]
    @property
    def _paramvec_bias(self):
        return self.param_bias[self.rtr_index, self.roi_index]
    @property
    def pred_size(self):
        if not self.hasfit:
            raise ValueError(
                "pred_size requested of model that has not been fit")
        sigma = self.param_cov
        eye = torch.eye(sigma.shape[0], dtype=sigma.dtype)
        # Precision of the population distribution.
        sigma_prec = torch.linalg.solve(sigma, eye)
        # Measurement precisions.
        dprec = []
        for r in range(len(self.param_bias)):
            logvar = torch.cat([
                self.param_rater_roi_logvar[r],
                self.param_rater_roi_logvar[r]])
            dprec.append(torch.diag(torch.exp(-logvar)))
        # Posterior covariance, shared by all subjects.
        precision = sigma_prec
        for d in dprec:
            precision = precision + d
        post_cov = torch.linalg.solve(precision, eye)
        # Posterior mean for each subject.
        post_mean = []
        for (ii, z) in enumerate(self.meas_size):
            h = sigma_prec @ self.param_mean
            for (r, d) in enumerate(dprec):
                lo = r * len(self.param_mean)
                hi = (r + 1) * len(self.param_mean)
                b = self.param_bias[r]
                b = torch.cat([b, b])
                h = h + d @ (z[lo:hi] - b)
            post_mean.append(post_cov @ h)
        return torch.stack(post_mean)
    @property
    def _predvec_size(self):
        return self.pred_size[self.sid_index, self._hemroi_index]
    @property
    def _predvec_meassize(self):
        return self._predvec_size + self._paramvec_bias
    def _df_index(self):
        idcols = list(type(self)._idcols)
        idcols.append('roi')
        idcs = self.meas_dataframe[idcols].values.T
        return dict(zip(idcols, idcs))
    @property
    def full_dataframe(self):
        return pd.DataFrame(
            dict(
                self._df_index(),
                meassize=self.measvec_size.detach().numpy(),
                truesize=self._predvec_size.detach().numpy(),
                predsize=self._predvec_meassize.detach().numpy(),
                bias=self._paramvec_bias.detach().numpy(),
                std=np.sqrt(np.exp(self._paramvec_logvar.detach().numpy()))))
    @property
    def pred_dataframe(self):
        return pd.DataFrame(
            dict(
                self._df_index(),
                size=self._predvec_meassize.detach().numpy()))

In [ ]:
lme_pct = ROISizesModel(dorsdf_meas, max_iter=2000, suffix='_percent', minvar=2.5e-6)
lme_mm2 = ROISizesModel(dorsdf_meas, max_iter=2000, suffix='_mm2', minvar=2.5e-2)

In [ ]:
# Column ordering so that they appear as lV3a, lV3b, lLO1, lIPS0, rV3a, rV3b, rLO1, rIPS0
colord = [2,3,1,0,6,7,5,4]
np.set_printoptions(suppress=True, linewidth=120)
print('='*32, 'mm2', '='*32)
r = lme_mm2.param_cov
sig = torch.diagonal(r)
r = r / torch.sqrt(sig[:,None] @ sig[None,:])
r = r.detach().numpy()
# VO1, VO2, hV4 sort like that alphabetically, so we unindex them
# into our usual ordering of hV4, VO1, VO2
r_mm2 = r[colord, :][:, colord]
print(r_mm2)
print('')

print('='*32, 'pct', '='*32)
r = lme_pct.param_cov
sig = torch.diagonal(r)
r = r / torch.sqrt(sig[:,None] @ sig[None,:])
r = r.detach().numpy()
r_pct = r[colord, :][:, colord]
print(r_pct)

### Report Dataframes

This cell builds up a number of useful dataframes for comparing model predictions and measurements.

In [ ]:
dorsdf_tall_pred_pct = lme_pct.meas_dataframe.rename(columns={'size': 'meassize'})
dorsdf_tall_pred_pct['predsize'] = lme_pct.pred_dataframe['size']
dorsdf_tall_pred_pct = dorsdf_tall_pred_pct.assign(unit='pct')

dorsdf_tall_pred_mm2 = lme_mm2.meas_dataframe.rename(columns={'size': 'meassize'})
dorsdf_tall_pred_mm2['predsize'] = lme_mm2.pred_dataframe['size']
dorsdf_tall_pred_mm2 = dorsdf_tall_pred_mm2.assign(unit='mm2')

dorsdf_tall = pd.concat([dorsdf_tall_pred_pct, dorsdf_tall_pred_mm2])
dorsdf_tall = dorsdf_tall[
    ['rater','sid','hemisphere','roi','unit','meassize','predsize']]
dorsdf = dorsdf_tall.pivot(
    index=['rater', 'sid', 'hemisphere'],
    columns=['roi', 'unit'],
    values=['meassize', 'predsize']).reset_index()
dorsdf_tall.columns = ['_'.join(ks) if not isinstance(ks, str) else ks for ks in dorsdf_tall.columns]
dorsdf.columns = ['_'.join(ks) if len(ks[1]) != 0 else ks[0] for ks in dorsdf.columns]
dorsdf

### Plots

This cell plots the predictions of all raters for all subjects versus their measurements.

In [ ]:
suffix = '_pct'
key = {'R1': 'k', 'R6': 'b', 'R7': 'g', 'R8': 'm', 'R9': 'y'}

(fig,axs) = plt.subplots(4, 2, figsize=(5,5), dpi=72*8, sharex=True, sharey=True)
fig.subplots_adjust(0,0,1,1,0.2,0.3)

for (h,axcol) in zip(('lh','rh'), axs.T):
    subdf = dorsdf[(dorsdf['hemisphere'] == h)]
    for (roi,ax) in zip(('V3a','V3b','LO1','IPS0'), axcol):
        roi = roi + suffix
        for (rater,color) in key.items():
            subdf_rr = subdf[subdf['rater'] == rater]
            ax.plot(
                subdf_rr['meassize_' + roi].values,
                subdf_rr['predsize_' + roi].values,
                color + 'o',
                alpha=0.25,
                label=rater)
        cat = np.concatenate(
            [subdf['predsize_' + roi].values,
             subdf['meassize_' + roi].values])
        mn = np.nanmin(cat)
        mx = np.nanmax(cat)
        ax.plot([mn,mx], [mn,mx], 'r--')
        ax.set_title(h.upper() + ' ' + roi[:-len(suffix)])
        if roi.startswith('VO2'):
            ax.set_xlabel('Measured Size [mm$^2$]')
        if h == 'lh':
            ax.set_ylabel('Predicted Size [mm$^2$]')

plt.show()

## Hierarchical ACE (Heritability) Model

Here, we combine our model above with an [ACE heritability model](https://en.wikipedia.org/wiki/ACE_model). The basic idea is similar to that described above: we postulate a 6x6 covariance matrix for the 6 ROIs (2 hemispheres x 3 ROIs each) over subjects, but because we have 5 raters, we observe 30 ROIs per subject. With twins, we have `twin-A` and `twin-B`, so we stack the 30 observations of each subject's ROIs with that of their twin, giving us a 60x60 covariance matrix. We build one of these for monozygotic twins $\mathbf{\Omega}_\text{MZ}$ and one for dizygotic twins $\mathbf{\Omega}_\text{DZ}$.

Previously, we tiled the full covariance matrix with the smaller covariance matrix plus a diagonal. Here, we expand the model a bit:
- We construct two 12x12 matrices of the *true* covariances between MZ and DZ twin pairs: $\mathbf{\Sigma}_\text{MZ}$ and $\mathbf{\Sigma}_\text{DZ}$; these contain 6 rows for the true ROIs of `twin-A` and 6 rows for `twin-B`. The diagonal 6x6 quadrants of these matrices are just $\mathbf{\Sigma}_\mathcal{Z}$ from the model above (because that's the covariance for all subjects with themselves). The off-diagonal 6x6 quadrants are the covariance matrices for pairs of MZ twins ($\mathbf{\Sigma}_\text{MZ}^{(\text{AB})}$) and DZ twins ($\mathbf{\Sigma}_\text{DZ}^{(\text{AB})}$).

  $$
  \mathbf{\Sigma}_\text{MZ} = \begin{pmatrix}
     \mathbf{\Sigma}_\mathcal{Z} & \mathbf{\Sigma}_\text{MZ}^{(\text{AB})} \\
     \mathbf{\Sigma}_\text{MZ}^{(\text{AB})} & \mathbf{\Sigma}_\mathcal{Z} \\
  \end{pmatrix}
  $$

  $$
  \mathbf{\Sigma}_\text{DZ} = \begin{pmatrix}
     \mathbf{\Sigma}_\mathcal{Z} & \mathbf{\Sigma}_\text{DZ}^{(\text{AB})} \\
     \mathbf{\Sigma}_\text{DZ}^{(\text{AB})} & \mathbf{\Sigma}_\mathcal{Z} \\
  \end{pmatrix}
  $$
  
- The *observed* covariance matrices $\mathbf{\Omega}_\text{MZ}$ and $\mathbf{\Omega}_\text{DZ}$ are built from these matrices:

  $$
  \mathbf{\Omega}_\text{MZ} = \begin{pmatrix}
     \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{1,:}^2) &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{2,:}^2) & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{3,:}^2) & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{4,:}^2) & 
         \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} \\
      \mathbf{\Sigma_\mathcal{Z}} & 
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{5,:}^2) &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} \\
      \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{1,:}^2)  & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{2,:}^2)  &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{3,:}^2)  & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{4,:}^2)  &
         \mathbf{\Sigma_\mathcal{Z}} \\
      \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} & \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} & \mathbf{\Sigma_\mathcal{Z}} &
         \mathbf{\Sigma_\mathcal{Z}} + \text{diag}(\tau_{5,:}^2)  \\
  \end{pmatrix}
  $$

  $\mathbf{\Omega}_\text{DZ}$ is constructed similarly, just using $\mathbf{\Sigma}_\text{DZ}^{(\text{AB})}$.
- For the ACE model, we have matrices $\mathbf{\Sigma}_\text{A}$, $\mathbf{\Sigma}_\text{C}$, and $\mathbf{\Sigma}_\text{E}$ that represent the shared genetic, shared environmental, and unique components of the covariance, respectively. We represent these in the model using Cholesky decomposition with lower triangular matrices $\mathbf{A}$, $\mathbf{C}$, and $\mathbf{E}$: $\mathbf{\Sigma}_\text{A} = \mathbf{A}\mathbf{A}^\intercal$, $\mathbf{\Sigma}_\text{C} = \mathbf{C}\mathbf{C}^\intercal$, $\mathbf{\Sigma}_\text{E} = \mathbf{E}\mathbf{E}^\intercal$. These explain our covariance as follows:

  $$
  \begin{eqnarray}
  \mathbf{\Sigma_\text{MZ}^{(\text{AB})}} &=& \mathbf{\Sigma}_\text{A} + \mathbf{\Sigma}_\text{C} \\
  \mathbf{\Sigma_\text{DZ}^{(\text{AB})}} &=& \mathbf{\Sigma}_\text{A}/2 + \mathbf{\Sigma}_\text{C} \\
  \mathbf{\Sigma_\mathcal{Z}} &=& \mathbf{\Sigma}_\text{A} + \mathbf{\Sigma}_\text{C} + \mathbf{\Sigma}_\text{E} \\
  \end{eqnarray}
  $$

We perform MLE on the above model using the twin pairs in the dataset below.

In [ ]:
class ROISizesACEModel(ROISizesModel):
    """A hierarchical ACE model that fits and uses the covariance matrix of
    twin ROI sizes across raters.

    This class implements a hierarchical linear mixed effects model and
    uses maximum likelihood optimization to obtain an estimate of the
    covariance matrix and mean size of the population of all subjects based
    on a the drawings of a set of raters.
    """
    _idcols = ('rater','sid','hemisphere')
    def __init__(self, df,
                 mz_pairs, dz_pairs,
                 suffix='_percent',
                 dtype=torch.float32,
                 lr=0.01,
                 max_iter=2000,
                 history_size=100,
                 minvar=0.025,
                 fit=True):
        # Run the parent class setup:
        super().__init__(
            df,
            suffix=suffix,
            dtype=dtype,
            lr=lr,
            max_iter=max_iter,
            history_size=history_size,
            minvar=minvar,
            fit=False)
        # We need to redeclare a few things; the superclass creates these parameters:
        # {'bias': self._param_bias,
        #  'logvar': self.param_rater_roi_logvar,
        #  'mean': self.param_mean,
        #  'logcovdiag': self._param_precov_logdiag,
        #  'covoffdiag': self._param_precov_offdiag}
        # We need some of these, but not logcovdiag or covoffdiag because we construct
        # these parameters from A, C, and E.
        del self.params['logcovdiag']
        del self.params['covoffdiag']
        # Some metadata:
        nmz = mz_pairs.shape[0]
        ndz = dz_pairs.shape[0]
        (nrtrs, nsids, nhems, nrois) = [
            self.counts[k] for k in ('raters', 'sids', 'hems', 'rois')]
        nrows_ace = nrois * nhems * 2  # these use `*2` due to tue twin pair
        ncols_full = nrois * nhems * nrtrs * 2
        self.sid_key = {sid: ii for (ii,sid) in enumerate(self.uniques['sids'])}
        mz_pairs = np.array([[self.sid_key[sid] for sid in u] for u in mz_pairs])
        dz_pairs = np.array([[self.sid_key[sid] for sid in u] for u in dz_pairs])
        # Make the A, C, E lower triangular parameter matrices:
        nels = nrois * nhems
        self._ace_shape = (nels, nels)
        self._ace_tril = tuple(torch.tril_indices(nels, nels, offset=0))
        nels = self._ace_tril[0].shape[0]
        self._param_trilA = torch.ones(nels, dtype=dtype, requires_grad=True)
        self._param_trilC = torch.ones(nels, dtype=dtype, requires_grad=True)
        self._param_trilE = torch.ones(nels, dtype=dtype, requires_grad=True)
        self.params['trilA'] = self._param_trilA
        self.params['trilC'] = self._param_trilC
        self.params['trilE'] = self._param_trilE
        # We now need MZ and DZ size matrices, separated for the twin types.
        self.mz_size = torch.zeros((nmz, ncols_full), dtype=dtype)
        self.dz_size = torch.zeros((ndz, ncols_full), dtype=dtype)
        ncols_half = ncols_full // 2
        for (ii,(a,b)) in enumerate(mz_pairs):
            self.mz_size[ii, :ncols_half] = self.meas_size[a, :]
            self.mz_size[ii, ncols_half:] = self.meas_size[b, :]
        for (ii,(a,b)) in enumerate(dz_pairs):
            self.dz_size[ii, :ncols_half] = self.meas_size[a, :]
            self.dz_size[ii, ncols_half:] = self.meas_size[b, :]
        self._mz_okrows = ~torch.any(torch.isnan(self.mz_size), dim=1)
        self._dz_okrows = ~torch.any(torch.isnan(self.dz_size), dim=1)
        # Now we run the fit!
        if fit:
            self.fit()
    def fit(self, lr=None, max_iter=None, history_size=None):
        """Fit the model to the data that was initially given.

        This function is typically run automatically by the constructor, but
        if the ``fit=False`` parameter is given to the constructor, then
        this function must be run prior to using and of the model's
        predictions.
        """
        # Parameters (default to the ones given in the constructor):
        lr = self.lr if lr is None else lr
        max_iter = self.max_iter if max_iter is None else max_iter
        history_size = self.history_size if history_size is None else history_size
        self.lr = lr
        self.max_iter = max_iter
        self.history_size = history_size
        # Set up the optimizer:
        optim = torch.optim.LBFGS(
            self.params.values(),
            lr=lr,
            max_iter=max_iter,
            line_search_fn='strong_wolfe',
            history_size=history_size)
        # Since the max iterations is given in the LBFGS optimizer, we don't have
        # a loop here, just a closure and a step:
        mz_meas_mtx = self.mz_size[self._mz_okrows, :]
        dz_meas_mtx = self.dz_size[self._dz_okrows, :]
        def closure():
            optim.zero_grad()
            mz_dist = dist.MultivariateNormal(
                loc=self.param_fullmean,
                covariance_matrix=self.param_mzfullcov)
            dz_dist = dist.MultivariateNormal(
                loc=self.param_fullmean,
                covariance_matrix=self.param_dzfullcov)
            loss = -(torch.sum(mz_dist.log_prob(mz_meas_mtx)) + torch.sum(dz_dist.log_prob(dz_meas_mtx)))
            loss.backward()
            return loss
        optim.step(closure)
        self.hasfit = True
        return optim
    # Note that param_bias is inherited.
    @property
    def param_fullmean(self):
        bias = self.param_bias
        nrtrs = bias.shape[0]
        bias = torch.flatten(torch.hstack([bias, bias]))
        mean0 = torch.cat([self.param_mean]*nrtrs) + bias
        return torch.cat([mean0, mean0])
    @property
    def param_mzfullcov(self):
        rtr_roi_var = torch.exp(self.param_rater_roi_logvar)
        nrtrs = self.counts['raters']
        rtr_roi_var = torch.flatten(torch.hstack([rtr_roi_var, rtr_roi_var]))
        diacov = torch.tile(self.param_cov, (nrtrs, nrtrs)) + torch.diag(rtr_roi_var)
        offcov = torch.tile(self.param_mzcov, (nrtrs, nrtrs))
        topcov = torch.cat([diacov, offcov], dim=1)
        botcov = torch.cat([offcov, diacov], dim=1)
        return torch.cat([topcov, botcov], dim=0)
    @property
    def param_dzfullcov(self):
        rtr_roi_var = torch.exp(self.param_rater_roi_logvar)
        nrtrs = self.counts['raters']
        rtr_roi_var = torch.flatten(torch.hstack([rtr_roi_var, rtr_roi_var]))
        diacov = torch.tile(self.param_cov, (nrtrs, nrtrs)) + torch.diag(rtr_roi_var)
        offcov = torch.tile(self.param_dzcov, (nrtrs, nrtrs))
        topcov = torch.cat([diacov, offcov], dim=1)
        botcov = torch.cat([offcov, diacov], dim=1)
        return torch.cat([topcov, botcov], dim=0)
    @property
    def param_mzpaircov(self):
        rtr_roi_var = torch.exp(self.param_rater_roi_logvar)
        nrtrs = self.counts['raters']
        rtr_roi_var = torch.flatten(torch.hstack([rtr_roi_var, rtr_roi_var]))
        topcov = torch.cat([self.param_cov, self.param_mzcov], dim=1)
        botcov = torch.cat([self.param_mzcov, self.param_cov], dim=1)
        return torch.cat([topcov, botcov], dim=0)
    @property
    def param_dzpaircov(self):
        rtr_roi_var = torch.exp(self.param_rater_roi_logvar)
        nrtrs = self.counts['raters']
        rtr_roi_var = torch.flatten(torch.hstack([rtr_roi_var, rtr_roi_var]))
        topcov = torch.cat([self.param_cov, self.param_dzcov], dim=1)
        botcov = torch.cat([self.param_dzcov, self.param_cov], dim=1)
        return torch.cat([topcov, botcov], dim=0)
    @property
    def param_mzcov(self):
        return self.param_covA + self.param_covC
    @property
    def param_dzcov(self):
        return self.param_covA/2 + self.param_covC
    @property
    def param_cov(self):
        return self.param_covA + self.param_covC + self.param_covE + self.minvar_diag
    @property
    def param_covA(self):
        M = torch.zeros(self._ace_shape, dtype=self.dtype)
        M[self._ace_tril] = self._param_trilA
        return M.T @ M
    @property
    def param_covC(self):
        M = torch.zeros(self._ace_shape, dtype=self.dtype)
        M[self._ace_tril] = self._param_trilC
        return M.T @ M
    @property
    def param_covE(self):
        M = torch.zeros(self._ace_shape, dtype=self.dtype)
        M[self._ace_tril] = self._param_trilE
        return M.T @ M
    @property
    def _paramvec_logvar(self):
        return self.param_rater_roi_logvar[self.rtr_index, self.roi_index]
    @property
    def _paramvec_bias(self):
        return self.param_bias[self.rtr_index, self.roi_index]
    @property
    def pred_size(self):
        raise ValueError("pred_size doesn't exist for the heritability model")
    @property
    def _predvec_size(self):
        return self.pred_size[self.sid_index, self._hemroi_index]
    @property
    def _predvec_meassize(self):
        return self._predvec_size + self._paramvec_bias
    def _df_index(self):
        idcols = list(type(self)._idcols)
        idcols.append('roi')
        idcs = self.meas_dataframe[idcols].values.T
        return dict(zip(idcols, idcs))
    @property
    def full_dataframe(self):
        return pd.DataFrame(
            dict(
                self._df_index(),
                meassize=self.measvec_size.detach().numpy(),
                truesize=self._predvec_size.detach().numpy(),
                predsize=self._predvec_meassize.detach().numpy(),
                bias=self._paramvec_bias.detach().numpy(),
                std=np.sqrt(np.exp(self._paramvec_logvar.detach().numpy()))))
    @property
    def pred_dataframe(self):
        return pd.DataFrame(
            dict(
                self._df_index(),
                size=self._predvec_meassize.detach().numpy()))

In [ ]:
mz_pairs = ny.data['hcp_lines'].retinotopy_sibling_pairs['monozygotic_twins']
dz_pairs = ny.data['hcp_lines'].retinotopy_sibling_pairs['dizygotic_twins']

fit = True

ace_pct = ROISizesACEModel(
    dorsdf_meas, mz_pairs, dz_pairs,
    max_iter=2000,
    suffix='_percent',
    minvar=2.5e-6,
    fit=fit)
ace_mm2 = ROISizesACEModel(
    dorsdf_meas, mz_pairs, dz_pairs,
    max_iter=2000,
    suffix='_mm2', 
    minvar=2.5e-2,
    fit=fit)

In [ ]:
np.set_printoptions(suppress=True, linewidth=160, precision=3)
# Column ordering so that they appear as lV3a, lV3b, lLO1, lIPS0, rV3a, rV3b, rLO1, rIPS0
# Without this, they sort alphabetically.
colord = [2,3,1,0,6,7,5,4]
colord = colord + [8 + k for k in colord]

def calc_rmtx(cov):
    r = cov
    sig = torch.diagonal(r)
    r = r / torch.sqrt(sig[:,None] @ sig[None,:])
    return r.detach().numpy()

print('='*32, 'mm2 MZ', '='*32)
r = calc_rmtx(ace_mm2.param_mzpaircov)
r = r[colord, :][:, colord]
print(r)
print('')

print('='*32, 'mm2 DZ', '='*32)
r = calc_rmtx(ace_mm2.param_dzpaircov)
r = r[colord, :][:, colord]
print(r)
print('')

print('='*119)
print('')

print('='*32, 'pct MZ', '='*32)
r = calc_rmtx(ace_pct.param_mzpaircov)
r = r[colord, :][:, colord]
print(r)
print('')

print('='*32, 'pct DZ', '='*32)
r = calc_rmtx(ace_pct.param_dzpaircov)
r = r[colord, :][:, colord]
print(r)

# There are 16 rows and 16 colums in the output matrices below;
# they store the estimates of the true correlation coefficients for MZ and DZ
# twin pairs in the dataset, ans the columns are in this order:
#  0.  Twin A LH V3a
#  1.  Twin A LH V3b
#  2.  Twin A LH LO1
#  3.  Twin A LH IPS0
#
#  4.  Twin A RH V3a
#  5.  Twin A RH V3b
#  6.  Twin A RH LO1
#  7.  Twin A RH IPS0
#--------------------
#  8.  Twin B LH V3a
#  9.  Twin B LH V3b
#  10. Twin B LH LO1
#  11. Twin B LH IPS0
#
#  12. Twin B RH V3a
#  13. Twin B RH V3b
#  14. Twin B RH LO1
#  15. Twin B RH IPS0

# So, for example, the correlation between the LH V3a of twin-A and RH V3b of
# twin-B is in cell [0, 13] and is equal to 0.296 for mm2 of MZ twins and
# 0.209 for percentages of MZ twins; for DZ twins these are 0.085 and 0.074.
# (Correlations are generally lower for the percentages because subjects with
# larget overall brains will have ROIs that are similarly larger overall,
# and this drives their correlation.